# 3. Calculate Eight Pre-Match Predictors

## Purpose

This notebook creates the eight explanatory variables for Linear
Regression 2.2.

The target variable is `goals_scored`: the number of goals scored by
one team in one match.

Four predictors describe the team's earlier tournament performance,
and four describe its opponent's earlier tournament performance.

All performance statistics exclude the current match. Opponent
statistics are obtained by matching each observation to the opponent's
own pre-match record for the same fixture.

In [1]:
import pandas as pd
from pathlib import Path

# Locate the project folder.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the validated team-match table from Notebook 2.
input_path = (
    PROJECT_ROOT / "data" / "processed" / "team_matches_208.csv"
)

assert input_path.is_file(), (
    "Run the saving cell in Notebook 2 before continuing."
)

team_matches = pd.read_csv(
    input_path,
    parse_dates=["match_date"]
)

# Arrange each team's matches chronologically.
team_matches = team_matches.sort_values(
    ["team", "match_date", "match_id"]
).reset_index(drop=True)

assert len(team_matches) == 208
assert not team_matches.duplicated(["match_id", "team"]).any()

display(team_matches.head())
print("Loaded team-match observations:", len(team_matches))

,match_id,stage,match_date,team,opponent,goals_scored,goals_conceded
0,19,Group stage,2026-06-16,Algeria,Argentina,0,3
1,44,Group stage,2026-06-22,Algeria,Jordan,2,1
2,72,Group stage,2026-06-27,Algeria,Austria,3,3
3,85,Round of 32,2026-07-02,Algeria,Switzerland,0,2
4,19,Group stage,2026-06-16,Argentina,Algeria,3,0


Loaded team-match observations: 208


## 3.1 Predictor Definitions

| Predictor | Meaning before kickoff |
|---|---|
| team_avg_goals_scored_before | Team's mean goals scored in earlier tournament matches |
| team_avg_goals_conceded_before | Team's mean goals conceded in earlier tournament matches |
| team_goal_difference_before | Team's cumulative goals scored minus goals conceded in earlier matches |
| team_win_rate_before | Proportion of earlier matches won on the match score |
| opponent_avg_goals_scored_before | Opponent's mean goals scored in earlier matches |
| opponent_avg_goals_conceded_before | Opponent's mean goals conceded in earlier matches |
| opponent_goal_difference_before | Opponent's cumulative goal difference in earlier matches |
| opponent_win_rate_before | Opponent's proportion of earlier matches won on the match score |

A penalty-shootout victory is not counted as a scoring win: a match
level after extra time remains a draw for this particular form measure.

Match IDs, dates, team names, and helper columns are not model predictors.

In [2]:
# These helper columns describe observed match outcomes.
# They are used only through earlier-match calculations, not directly
# as predictors of the same match.
team_matches["match_goal_difference"] = (
    team_matches["goals_scored"] - team_matches["goals_conceded"]
)

team_matches["scoring_win"] = (
    team_matches["goals_scored"] > team_matches["goals_conceded"]
).astype(int)

# Count how many matches each team had played before each fixture.
team_matches["prior_matches"] = (
    team_matches.groupby("team").cumcount()
)

display(team_matches[[
    "team",
    "match_date",
    "goals_scored",
    "goals_conceded",
    "prior_matches"
]].head(10))

,team,match_date,goals_scored,goals_conceded,prior_matches
0,Algeria,2026-06-16,0,3,0
1,Algeria,2026-06-22,2,1,1
2,Algeria,2026-06-27,3,3,2
3,Algeria,2026-07-02,0,2,3
4,Argentina,2026-06-16,3,0,0
5,Argentina,2026-06-22,2,0,1
6,Argentina,2026-06-27,3,1,2
7,Argentina,2026-07-03,3,2,3
8,Argentina,2026-07-07,3,2,4
9,Argentina,2026-07-11,3,1,5


## 3.2 Calculate the Team's Historical Form

For each team, shift(1) moves the outcomes back by one match. Expanding
calculations then summarise only the matches played earlier.

The current match is excluded from its own predictors.

Opening matches have no earlier tournament history, so their historical
averages are initially missing.

In [3]:
# Calculate average goals scored using earlier matches only.
team_matches["team_avg_goals_scored_before"] = (
    team_matches.groupby("team")["goals_scored"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

# Calculate average goals conceded using earlier matches only.
team_matches["team_avg_goals_conceded_before"] = (
    team_matches.groupby("team")["goals_conceded"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

# Calculate cumulative goal difference using earlier matches only.
team_matches["team_goal_difference_before"] = (
    team_matches.groupby("team")["match_goal_difference"]
    .transform(lambda s: s.shift(1).expanding().sum())
)

# Calculate the proportion of earlier matches won on the match score.
team_matches["team_win_rate_before"] = (
    team_matches.groupby("team")["scoring_win"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

team_predictors = [
    "team_avg_goals_scored_before",
    "team_avg_goals_conceded_before",
    "team_goal_difference_before",
    "team_win_rate_before"
]

display(team_matches[[
    "team", "match_date", "prior_matches"
] + team_predictors].head(10))

,team,match_date,prior_matches,team_avg_goals_scored_before,team_avg_goals_conceded_before,team_goal_difference_before,team_win_rate_before
0,Algeria,2026-06-16,0,NaN,NaN,NaN,NaN
1,Algeria,2026-06-22,1,0.000000,3.000000,-3.0,0.000000
2,Algeria,2026-06-27,2,1.000000,2.000000,-2.0,0.500000
3,Algeria,2026-07-02,3,1.666667,2.333333,-2.0,0.333333
4,Argentina,2026-06-16,0,NaN,NaN,NaN,NaN
5,Argentina,2026-06-22,1,3.000000,0.000000,3.0,1.000000
6,Argentina,2026-06-27,2,2.500000,0.000000,5.0,1.000000
7,Argentina,2026-07-03,3,2.666667,0.333333,7.0,1.000000
8,Argentina,2026-07-07,4,2.750000,0.750000,8.0,1.000000
9,Argentina,2026-07-11,5,2.800000,1.000000,9.0,1.000000


## 3.3 Handle Teams' Opening Matches

Every team begins the tournament without a tournament performance
history.

We use a fixed zero-history convention: all four historical-form
predictors are set to zero for a team's opening match.

These zeros mean "no previous tournament history", not evidence that
the team has zero attacking ability. This convention preserves all
208 observations and uses no future match results.

A limitation is that all teams have identical form predictors in their
opening matches. The model therefore cannot distinguish their opening
match strength using these variables.

In [4]:
# Identify observations with no earlier tournament matches.
opening_match = team_matches["prior_matches"].eq(0)

# Only opening-match rows should have missing historical predictors.
assert team_matches.loc[
    opening_match, team_predictors
].isna().all().all()

assert team_matches.loc[
    ~opening_match, team_predictors
].notna().all().all()

# Use fixed values rather than averages derived from future results.
team_matches.loc[opening_match, team_predictors] = 0.0

print("Opening team-match observations:", opening_match.sum())
print(
    "Missing team predictor values:",
    team_matches[team_predictors].isna().sum().sum()
)

Opening team-match observations: 48
Missing team predictor values: 0


## 3.4 Match Each Team to Its Opponent's Pre-Match Form

The opponent's predictors must describe the opponent's own earlier
performances.

We create a lookup table from the team predictors already calculated.
We then join it using the match ID and opponent name.

This avoids calculating opponent form from the wrong team's scores.

In [5]:
# Map each team predictor to its equivalent opponent predictor.
opponent_names = {
    "team_avg_goals_scored_before":
        "opponent_avg_goals_scored_before",
    "team_avg_goals_conceded_before":
        "opponent_avg_goals_conceded_before",
    "team_goal_difference_before":
        "opponent_goal_difference_before",
    "team_win_rate_before":
        "opponent_win_rate_before"
}

# Build a lookup containing each team's own pre-match form.
opponent_lookup = team_matches[[
    "match_id", "team"
] + team_predictors].copy()

# Rename team to opponent so the lookup matches the opposing side.
opponent_lookup = opponent_lookup.rename(
    columns={"team": "opponent", **opponent_names}
)

# Attach the opponent's form for the same fixture.
features = team_matches.merge(
    opponent_lookup,
    on=["match_id", "opponent"],
    how="left",
    validate="one_to_one"
)

opponent_predictors = list(opponent_names.values())
predictors = team_predictors + opponent_predictors

display(features[[
    "match_id", "team", "opponent"
] + predictors].head(6))

,match_id,team,opponent,team_avg_goals_scored_before,team_avg_goals_conceded_before,team_goal_difference_before,team_win_rate_before,opponent_avg_goals_scored_before,opponent_avg_goals_conceded_before,opponent_goal_difference_before,opponent_win_rate_before
0,19,Algeria,Argentina,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000
1,44,Algeria,Jordan,0.000000,3.000000,-3.0,0.000000,1.000000,3.0,-2.0,0.000000
2,72,Algeria,Austria,1.000000,2.000000,-2.0,0.500000,1.500000,1.5,0.0,0.500000
3,85,Algeria,Switzerland,1.666667,2.333333,-2.0,0.333333,2.333333,1.0,4.0,0.666667
4,19,Argentina,Algeria,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000
5,41,Argentina,Austria,3.000000,0.000000,3.0,1.000000,3.000000,1.0,2.0,1.000000


## 3.5 Check an Example Against Earlier Results

Mexico won its opening match 2–0 against South Africa.

Before Mexico's second match, its historical predictors should be:

- Average goals scored: 2
- Average goals conceded: 0
- Cumulative goal difference: 2
- Scoring win rate: 1

We inspect Mexico's records to confirm the calculations.

In [6]:
# Display Mexico's matches in chronological order.
mexico = features.loc[
    features["team"].eq("Mexico")
].sort_values(["match_date", "match_id"])

display(mexico[[
    "match_date",
    "team",
    "opponent",
    "goals_scored",
    "prior_matches"
] + team_predictors])

# Verify the pre-match values for Mexico's second match.
second_match = mexico.iloc[1]

assert second_match["team_avg_goals_scored_before"] == 2
assert second_match["team_avg_goals_conceded_before"] == 0
assert second_match["team_goal_difference_before"] == 2
assert second_match["team_win_rate_before"] == 1

print("Mexico's second-match predictor check passed.")

,match_date,team,opponent,goals_scored,prior_matches,team_avg_goals_scored_before,team_avg_goals_conceded_before,team_goal_difference_before,team_win_rate_before
119,2026-06-11,Mexico,South Africa,2,0,0.0,0.0,0.0,0.0
120,2026-06-18,Mexico,Korea Republic,1,1,2.0,0.0,2.0,1.0
121,2026-06-24,Mexico,Czechia,3,2,1.5,0.0,3.0,1.0
122,2026-06-30,Mexico,Ecuador,2,3,2.0,0.0,6.0,1.0
123,2026-07-05,Mexico,England,2,4,2.0,0.0,8.0,1.0


Mexico's second-match predictor check passed.


## 3.6 Validate and Save the Modelling Dataset

The final dataset must contain:

- 208 team-match observations
- 104 unique matches
- Exactly eight explanatory variables
- One target variable: goals_scored
- No missing values in the modelling columns

Identifiers and descriptive fields are retained for traceability but
will not be included in the model's explanatory-variable matrix.

The predictors may be correlated because they describe related aspects
of performance. Correlation and model diagnostics will be examined
before interpreting regression coefficients.

In [7]:
# Keep identifiers, descriptive fields, the target, and eight predictors.
final_columns = [
    "match_id",
    "stage",
    "match_date",
    "team",
    "opponent",
    "goals_scored"
] + predictors

final_df = features[final_columns].copy()

final_df = final_df.sort_values(
    ["match_date", "match_id", "team"]
).reset_index(drop=True)

# Validate the final modelling table.
assert len(final_df) == 208
assert final_df["match_id"].nunique() == 104
assert len(predictors) == 8
assert not final_df.duplicated(["match_id", "team"]).any()
assert final_df[predictors + ["goals_scored"]].notna().all().all()

# Validate the bounds of the win-rate predictors.
for column in [
    "team_win_rate_before",
    "opponent_win_rate_before"
]:
    assert final_df[column].between(0, 1).all()

# Save the final modelling dataset.
output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "worldcup2026_regression_2_2.csv"
)

final_df.to_csv(output_path, index=False)

print("Rows:", len(final_df))
print("Unique matches:", final_df["match_id"].nunique())
print("Explanatory variables:", len(predictors))
print(
    "Missing modelling values:",
    final_df[predictors + ["goals_scored"]].isna().sum().sum()
)
print("Dataset dimensions:", final_df.shape)
print("Saved:", output_path.name)

Rows: 208
Unique matches: 104
Explanatory variables: 8
Missing modelling values: 0
Dataset dimensions: (208, 14)
Saved: worldcup2026_regression_2_2.csv
